**Deutsch's algorithm.**
Deutsch's algorithm (1985) is the first quantum algorithm to demonstrate
a provable speedup over any classical approach.
Given a black-box function $f:\{0,1\}\to\{0,1\}$, it determines whether
$f$ is **constant** ($f(0)=f(1)$) or **balanced** ($f(0)\neq f(1)$)
using only **one** query to $f$, whereas any classical algorithm requires two.

There are exactly four possible functions:

| Name | $f(0)$ | $f(1)$ | Type |
|---|---|---|---|
| NOP | 0 | 1 | Balanced |
| NOT | 1 | 0 | Balanced |
| Always 0 | 0 | 0 | Constant |
| Always 1 | 1 | 1 | Constant |

**How the algorithm works.**
The oracle acts directly on q0, modifying its state according to $f$.
The ancilla qubit q1 is initialized to $|1\rangle$ and put into $|-\rangle$
by a Hadamard, serving as the phase kickback register.
The full circuit is:

1. Initialize q1 $= |1\rangle$
2. Apply H to both qubits: $|0\rangle \to |{+}\rangle$, $|1\rangle \to |-\rangle$
3. Apply the oracle (acts on q0)
4. Apply H to q0
5. Measure q0: **0 $\Rightarrow$ balanced, 1 $\Rightarrow$ constant**

Note: this notebook uses the convention 0=balanced, 1=constant,
which is the opposite of the standard textbook Deutsch convention
(where 0=constant, 1=balanced) due to the direct-action oracle formulation used here.

---
**Cells 01-08** verify each oracle classically (no superposition).
**Cells 09-12** run the full Deutsch detector with one shot.

A single `AerSimulator` backend is shared across all cells.

In [ ]:
"""deutsch.ipynb"""

# Cell 01 - Imports and shared AerSimulator backend

from IPython.display import Math, display
from qiskit import QuantumCircuit, transpile
from qiskit.visualization import plot_distribution
from qiskit_aer import AerSimulator

from qis101_utils import as_latex

backend = AerSimulator()

print(f"Backend name     : {backend.name}")
print(f"Aer version      : {backend.configuration().backend_version}")
print(f"Max qubits       : {backend.configuration().n_qubits}")
print(f"Max shots        : {backend.configuration().max_shots:,}")
methods = backend.available_methods()
if methods:
    print("Available methods:")
    for method in methods:
        print(f"  {method}")


---
**Cell 02 - NOP oracle acting on classical input q0=$|0\rangle$.**
The NOP oracle implements $f(x)=x$ (balanced): it leaves q0 unchanged.
With q0=$|0\rangle$, identity gates on both qubits produce no state change.
Measurement of q0 always returns 0.

In [ ]:
# Cell 02 - NOP oracle (f(x)=x, balanced), classical input q0=|0>
# Oracle: id(0), id(1) -> no change to state
# Measurement always returns 0

qc = QuantumCircuit(2, 1)
qc.save_statevector("sv1")
qc.save_statevector("sv2")
qc.id(0)
qc.id(1)
qc.save_statevector("sv3")
qc.measure(0, 0)

display(qc.draw(output="mpl"))
result = backend.run(transpile(qc, backend), shots=100_000).result()

sv1 = result.data(0)["sv1"]
sv2 = result.data(0)["sv2"]
sv3 = result.data(0)["sv3"]

display(as_latex(sv1, prefix=r"\mathbf{Statevector\;1}="))
display(as_latex(sv2, prefix=r"\mathbf{Statevector\;2}="))
display(as_latex(sv3, prefix=r"\mathbf{Statevector\;3}="))
display(plot_distribution(result.get_counts(qc)))

---
**Cell 03 - NOP oracle acting on classical input q0=$|1\rangle$.**
Same NOP oracle ($f(x)=x$) but with q0 initialized to $|1\rangle$ via X.
The identity oracle leaves q0 unchanged, so measurement always returns 1.

In [ ]:
# Cell 03 - NOP oracle (f(x)=x, balanced), classical input q0=|1>
# Oracle: id(0), id(1) -> q0 stays |1>
# Measurement always returns 1

qc = QuantumCircuit(2, 1)
qc.x(0)
qc.save_statevector("sv1")
qc.save_statevector("sv2")
qc.id(0)
qc.id(1)
qc.save_statevector("sv3")
qc.measure(0, 0)

display(qc.draw(output="mpl"))
result = backend.run(transpile(qc, backend), shots=100_000).result()

sv1 = result.data(0)["sv1"]
sv2 = result.data(0)["sv2"]
sv3 = result.data(0)["sv3"]

display(as_latex(sv1, prefix=r"\mathbf{Statevector\;1}="))
display(as_latex(sv2, prefix=r"\mathbf{Statevector\;2}="))
display(as_latex(sv3, prefix=r"\mathbf{Statevector\;3}="))
display(plot_distribution(result.get_counts(qc)))

---
**Cell 04 - Always-0 oracle acting on classical input q0=$|0\rangle$.**
The Always-0 oracle implements $f(x)=0$ (constant).
The two-CNOT sequence CX(0,1) then CX(1,0) achieves this:
for q0=$|0\rangle$, q1 is unchanged by CX(0,1) and q0 is unchanged by CX(1,0),
so the output is still $|00\rangle$ and q0 measures 0.
For q0=$|1\rangle$ (Cell 05), CX(0,1) flips q1 to 1, then CX(1,0) flips q0
back to 0 - output q0 is always 0 regardless of input.

In [ ]:
# Cell 04 - Always-0 oracle (f(x)=0, constant), classical input q0=|0>
# Oracle: CX(0->1), CX(1->0) -> q0 always 0 regardless of input
# q0=0: CX01 no-op, CX10 no-op -> output 0

qc = QuantumCircuit(2, 1)
qc.save_statevector("sv1")
qc.save_statevector("sv2")
qc.cx(0, 1)
qc.cx(1, 0)
qc.save_statevector("sv3")
qc.measure(0, 0)

display(qc.draw(output="mpl"))
result = backend.run(transpile(qc, backend), shots=100_000).result()

sv1 = result.data(0)["sv1"]
sv2 = result.data(0)["sv2"]
sv3 = result.data(0)["sv3"]

display(as_latex(sv1, prefix=r"\mathbf{Statevector\;1}="))
display(as_latex(sv2, prefix=r"\mathbf{Statevector\;2}="))
display(as_latex(sv3, prefix=r"\mathbf{Statevector\;3}="))
display(plot_distribution(result.get_counts(qc)))

---
**Cell 05 - Always-0 oracle acting on classical input q0=$|1\rangle$.**
CX(0,1) flips q1 to $|1\rangle$ (since q0=1), then CX(1,0) flips q0 back to $|0\rangle$.
Output q0 = 0 for input q0 = 1, confirming $f(1)=0$.
Both inputs map to 0 - the function is constant.

In [ ]:
# Cell 05 - Always-0 oracle (f(x)=0, constant), classical input q0=|1>
# q0=1: CX01 flips q1 to 1, CX10 flips q0 back to 0 -> output 0

qc = QuantumCircuit(2, 1)
qc.x(0)
qc.save_statevector("sv1")
qc.save_statevector("sv2")
qc.cx(0, 1)
qc.cx(1, 0)
qc.save_statevector("sv3")
qc.measure(0, 0)

display(qc.draw(output="mpl"))
result = backend.run(transpile(qc, backend), shots=100_000).result()

sv1 = result.data(0)["sv1"]
sv2 = result.data(0)["sv2"]
sv3 = result.data(0)["sv3"]

display(as_latex(sv1, prefix=r"\mathbf{Statevector\;1}="))
display(as_latex(sv2, prefix=r"\mathbf{Statevector\;2}="))
display(as_latex(sv3, prefix=r"\mathbf{Statevector\;3}="))
display(plot_distribution(result.get_counts(qc)))

---
**Cell 06 - Always-1 oracle acting on classical input q0=$|0\rangle$.**
The Always-1 oracle adds X(q0) after the Always-0 circuit,
flipping q0 from 0 to 1 at the end.
For q0=$|0\rangle$ input: the two CNOTs leave q0=0, then X flips to 1.
Output is always 1 regardless of input - confirmed as constant.

In [ ]:
# Cell 06 - Always-1 oracle (f(x)=1, constant), classical input q0=|0>
# Oracle: CX(0->1), CX(1->0), X(0) -> q0 always 1 regardless of input

qc = QuantumCircuit(2, 1)
qc.save_statevector("sv1")
qc.save_statevector("sv2")
qc.cx(0, 1)
qc.cx(1, 0)
qc.x(0)
qc.save_statevector("sv3")
qc.measure(0, 0)

display(qc.draw(output="mpl"))
result = backend.run(transpile(qc, backend), shots=100_000).result()

sv1 = result.data(0)["sv1"]
sv2 = result.data(0)["sv2"]
sv3 = result.data(0)["sv3"]

display(as_latex(sv1, prefix=r"\mathbf{Statevector\;1}="))
display(as_latex(sv2, prefix=r"\mathbf{Statevector\;2}="))
display(as_latex(sv3, prefix=r"\mathbf{Statevector\;3}="))
display(plot_distribution(result.get_counts(qc)))

---
**Cell 07 - Always-1 oracle acting on classical input q0=$|1\rangle$.**
For q0=$|1\rangle$ input: CX(0,1) flips q1, CX(1,0) resets q0 to 0,
then X flips q0 to 1.
Both inputs give output 1, confirming $f$ is constant.

In [ ]:
# Cell 07 - Always-1 oracle (f(x)=1, constant), classical input q0=|1>
# CX01 flips q1, CX10 resets q0 to 0, X flips q0 to 1 -> output always 1

qc = QuantumCircuit(2, 1)
qc.save_statevector("sv1")
qc.save_statevector("sv2")
qc.x(0)
qc.id(1)
qc.save_statevector("sv3")
qc.measure(0, 0)

display(qc.draw(output="mpl"))
result = backend.run(transpile(qc, backend), shots=100_000).result()

sv1 = result.data(0)["sv1"]
sv2 = result.data(0)["sv2"]
sv3 = result.data(0)["sv3"]

display(as_latex(sv1, prefix=r"\mathbf{Statevector\;1}="))
display(as_latex(sv2, prefix=r"\mathbf{Statevector\;2}="))
display(as_latex(sv3, prefix=r"\mathbf{Statevector\;3}="))
display(plot_distribution(result.get_counts(qc)))

---
**Cell 08 - NOT oracle acting on classical input q0=$|0\rangle$.**
The NOT oracle implements $f(x) = \lnot x$ (balanced): X flips q0.
For q0=$|0\rangle$: X maps to $|1\rangle$, so $f(0)=1$.

In [ ]:
# Cell 08 - NOT oracle (f(x)=NOT x, balanced), classical input q0=|0>
# Oracle: X(0) -> flips q0; f(0)=1

qc = QuantumCircuit(2, 1)
qc.save_statevector("sv1")
qc.save_statevector("sv2")
qc.x(0)
qc.id(1)
qc.save_statevector("sv3")
qc.measure(0, 0)

display(qc.draw(output="mpl"))
result = backend.run(transpile(qc, backend), shots=100_000).result()

sv1 = result.data(0)["sv1"]
sv2 = result.data(0)["sv2"]
sv3 = result.data(0)["sv3"]

display(as_latex(sv1, prefix=r"\mathbf{Statevector\;1}="))
display(as_latex(sv2, prefix=r"\mathbf{Statevector\;2}="))
display(as_latex(sv3, prefix=r"\mathbf{Statevector\;3}="))
display(plot_distribution(result.get_counts(qc)))

---
**Cell 09 - NOT oracle acting on classical input q0=$|1\rangle$.**
For q0=$|1\rangle$: X maps to $|0\rangle$, so $f(1)=0$.
Together $f(0)=1, f(1)=0$ confirms NOT is balanced.

In [ ]:
# Cell 09 - NOT oracle (f(x)=NOT x, balanced), classical input q0=|1>
# X(0) flips q0: |1> -> |0>; f(1)=0

qc = QuantumCircuit(2, 1)
qc.x(0)
qc.save_statevector("sv1")
qc.save_statevector("sv2")
qc.x(0)
qc.id(1)
qc.save_statevector("sv3")
qc.measure(0, 0)

display(qc.draw(output="mpl"))
result = backend.run(transpile(qc, backend), shots=100_000).result()

sv1 = result.data(0)["sv1"]
sv2 = result.data(0)["sv2"]
sv3 = result.data(0)["sv3"]

display(as_latex(sv1, prefix=r"\mathbf{Statevector\;1}="))
display(as_latex(sv2, prefix=r"\mathbf{Statevector\;2}="))
display(as_latex(sv3, prefix=r"\mathbf{Statevector\;3}="))
display(plot_distribution(result.get_counts(qc)))

---
**Deutsch detector - Cells 10-13.**
The detector wraps each oracle type in the full Deutsch circuit:
X(q1) $\to$ H(q0,q1) $\to$ oracle $\to$ H(q0) $\to$ measure q0.
Because the oracle acts on q0 while q1 is in $|-\rangle$, phase kickback
transfers the oracle's character to q0.
After the final H, q0 collapses to:

$$q_0 = 0 \Rightarrow \text{Balanced} \qquad q_0 = 1 \Rightarrow \text{Constant}$$

Only **one shot** is needed - a single query suffices to determine the answer with certainty.

---
**Cell 10 - Deutsch detector: NOP oracle (Balanced).**
The NOP oracle (identity) applies no gates between the two H layers.
Phase kickback via the $|-\rangle$ ancilla causes q0 to return to $|0\rangle$
after the final H, yielding measurement 0 $\Rightarrow$ Balanced.

In [ ]:
# Cell 10 - Deutsch detector: NOP oracle (f(x)=x, balanced)
# Circuit: X(q1), H(q0,q1), id(q0,q1), H(q0), measure q0
# Expected: measures 0 -> Balanced

qc = QuantumCircuit(2, 1)
qc.x(1)
qc.save_statevector("sv1")
qc.h(0)
qc.h(1)
qc.save_statevector("sv2")
qc.id(0)
qc.id(1)
qc.save_statevector("sv3")
qc.h(0)
qc.save_statevector("sv4")
qc.measure(0, 0)

display(qc.draw(output="mpl"))
result = backend.run(transpile(qc, backend), shots=1).result()

sv1 = result.data(0)["sv1"]
sv2 = result.data(0)["sv2"]
sv3 = result.data(0)["sv3"]
sv4 = result.data(0)["sv4"]

display(as_latex(sv1, prefix=r"\mathbf{Statevector\;1}="))
display(as_latex(sv2, prefix=r"\mathbf{Statevector\;2}="))
display(as_latex(sv3, prefix=r"\mathbf{Statevector\;3}="))
display(as_latex(sv4, prefix=r"\mathbf{Statevector\;4}="))

counts = result.get_counts(qc)
display(plot_distribution(counts))

if counts.get("0") == 1:
    display(Math(r"\large\color{green}{\mathrm{\mathbf{Balanced}}}"))
if counts.get("1") == 1:
    display(Math(r"\large\color{blue}{\mathrm{\mathbf{Constant}}}"))

---
**Cell 11 - Deutsch detector: Always-0 oracle (Constant).**
The Always-0 oracle (CX(0,1), CX(1,0)) maps both inputs to 0.
The phase kickback is symmetric across both input branches,
leaving q0 in $|1\rangle$ after the final H: measures 1 $\Rightarrow$ Constant.

In [ ]:
# Cell 11 - Deutsch detector: Always-0 oracle (f(x)=0, constant)
# Circuit: X(q1), H(q0,q1), CX01, CX10, H(q0), measure q0
# Expected: measures 1 -> Constant

qc = QuantumCircuit(2, 1)
qc.x(1)
qc.save_statevector("sv1")
qc.h(0)
qc.h(1)
qc.save_statevector("sv2")
qc.cx(0, 1)
qc.cx(1, 0)
qc.save_statevector("sv3")
qc.h(0)
qc.save_statevector("sv4")
qc.measure(0, 0)

display(qc.draw(output="mpl"))
result = backend.run(transpile(qc, backend), shots=1).result()

sv1 = result.data(0)["sv1"]
sv2 = result.data(0)["sv2"]
sv3 = result.data(0)["sv3"]
sv4 = result.data(0)["sv4"]

display(as_latex(sv1, prefix=r"\mathbf{Statevector\;1}="))
display(as_latex(sv2, prefix=r"\mathbf{Statevector\;2}="))
display(as_latex(sv3, prefix=r"\mathbf{Statevector\;3}="))
display(as_latex(sv4, prefix=r"\mathbf{Statevector\;4}="))

counts = result.get_counts(qc)
display(plot_distribution(counts))

if counts.get("0") == 1:
    display(Math(r"\large\color{green}{\mathrm{\mathbf{Balanced}}}"))
if counts.get("1") == 1:
    display(Math(r"\large\color{blue}{\mathrm{\mathbf{Constant}}}"))

---
**Cell 12 - Deutsch detector: Always-1 oracle (Constant).**
The Always-1 oracle adds X(q0) after the Always-0 circuit.
The extra X changes the phase of the kickback but the function remains
constant, so q0 again measures 1 $\Rightarrow$ Constant.

In [ ]:
# Cell 12 - Deutsch detector: Always-1 oracle (f(x)=1, constant)
# Circuit: X(q1), H(q0,q1), CX01, CX10, X(q0), H(q0), measure q0
# Expected: measures 1 -> Constant

qc = QuantumCircuit(2, 1)
qc.x(1)
qc.save_statevector("sv1")
qc.h(0)
qc.h(1)
qc.save_statevector("sv2")
qc.cx(0, 1)
qc.cx(1, 0)
qc.x(0)
qc.save_statevector("sv3")
qc.h(0)
qc.save_statevector("sv4")
qc.measure(0, 0)

display(qc.draw(output="mpl"))
result = backend.run(transpile(qc, backend), shots=1).result()

sv1 = result.data(0)["sv1"]
sv2 = result.data(0)["sv2"]
sv3 = result.data(0)["sv3"]
sv4 = result.data(0)["sv4"]

display(as_latex(sv1, prefix=r"\mathbf{Statevector\;1}="))
display(as_latex(sv2, prefix=r"\mathbf{Statevector\;2}="))
display(as_latex(sv3, prefix=r"\mathbf{Statevector\;3}="))
display(as_latex(sv4, prefix=r"\mathbf{Statevector\;4}="))

counts = result.get_counts(qc)
display(plot_distribution(counts))

if counts.get("0") == 1:
    display(Math(r"\large\color{green}{\mathrm{\mathbf{Balanced}}}"))
if counts.get("1") == 1:
    display(Math(r"\large\color{blue}{\mathrm{\mathbf{Constant}}}"))

---
**Cell 13 - Deutsch detector: NOT oracle (Balanced).**
The NOT oracle applies X(q0) directly.
Since $f(x)=\lnot x$ is balanced, q0 returns to $|0\rangle$ after
the final H: measures 0 $\Rightarrow$ Balanced.

In [ ]:
# Cell 13 - Deutsch detector: NOT oracle (f(x)=NOT x, balanced)
# Circuit: X(q1), H(q0,q1), X(q0), H(q0), measure q0
# Expected: measures 0 -> Balanced

qc = QuantumCircuit(2, 1)
qc.x(1)
qc.save_statevector("sv1")
qc.h(0)
qc.h(1)
qc.save_statevector("sv2")
qc.x(0)
qc.id(1)
qc.save_statevector("sv3")
qc.h(0)
qc.save_statevector("sv4")
qc.measure(0, 0)

display(qc.draw(output="mpl"))
result = backend.run(transpile(qc, backend), shots=1).result()

sv1 = result.data(0)["sv1"]
sv2 = result.data(0)["sv2"]
sv3 = result.data(0)["sv3"]
sv4 = result.data(0)["sv4"]

display(as_latex(sv1, prefix=r"\mathbf{Statevector\;1}="))
display(as_latex(sv2, prefix=r"\mathbf{Statevector\;2}="))
display(as_latex(sv3, prefix=r"\mathbf{Statevector\;3}="))
display(as_latex(sv4, prefix=r"\mathbf{Statevector\;4}="))

counts = result.get_counts(qc)
display(plot_distribution(counts))

if counts.get("0") == 1:
    display(Math(r"\large\color{green}{\mathrm{\mathbf{Balanced}}}"))
if counts.get("1") == 1:
    display(Math(r"\large\color{blue}{\mathrm{\mathbf{Constant}}}"))